<a href="https://colab.research.google.com/github/Kamalashrinithi19/kamalashrinithi-codeboosters-2026/blob/main/Day9/Mini_Project_Day9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq -q
import sqlite3
import os
import pandas as pd
import re
from groq import Groq
print("All libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.6 MB/s eta 0:00:00
All libraries imported successfully!


In [ ]:
# option 1: set your Groq API here

os.environ['GROQ_API_KEY'] = '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'
client = Groq(api_key=os.environ['GROQ_API_KEY'])

#insert model
MODEL = 'llama-3.1-8b-instant'
print("griq client initialised")
print(f"Using model: {MODEL}")

griq client initialised
Using model: llama-3.1-8b-instant


In [ ]:
import io # to  show input and output

df=pd.read_csv('student_performance.csv')
print("Loaded dataset successfully!")

print("Shape of Dataset:",df.shape[0], "rows", df.shape[1],"columns" )
print("First 5 rows of dataset:")
print(df.head())

Loaded dataset successfully!
Shape of Dataset: 30 rows 13 columns
First 5 rows of dataset:
   student_id          name  age  gender        department  semester  \
0        1001  Aarav Sharma   19    Male  Computer Science         2   
1        1002   Priya Patel   20  Female  Computer Science         2   
2        1003   Rohit Verma   19    Male       Electronics         2   
3        1004   Sneha Reddy   20  Female        Mechanical         2   
4        1005    Arjun Nair   19    Male  Computer Science         2   

   math_score  science_score  english_score  programming_score  \
0          85             78             72                 91   
1          76             82             88                 79   
2          65             74             61                 55   
3          70             80             75                 48   
4          92             88             81                 95   

   attendance_percentage       city  admission_year  
0                     92 

In [ ]:
conn = sqlite3.connect('college.db')      # creating a database conection
df.to_sql("students", conn, if_exists="replace", index=False)
print("SQLite databse loaded with dataset successfully!")

# Test db connection
test_df=pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students", conn)
print(f"Verification {test_df['total_rows'][0]} rows in the database")

SQLite databse loaded with dataset successfully!
Verification 30 rows in the database


In [ ]:
# Function to get the database schema

def get_schema(conn, table_name="students"):
  """
  Reads structure of database...
  PARAMS:
    conn : DB connection through which we accessing the table
    table_name :  name of the table we need to access now
  """

  # Query SQLite's internal table info

  cursor = conn.cursor()    # connection to SQLite used to execute SQL Query
  cursor.execute(f"PRAGMA table_info({table_name})")   # PRAGMA table_info is built-in python function to show all column names of the table
# PRAGMA table_info - a specail SQLite command that returns column names
  columns = cursor.fetchall()
  # returns a list of tuples, one tuple per column
  # Example tuple : (0, 'student_id', 'INTEGER', 0, None, ...)
  # build a human-readable schema description
  schema_lines = [f"Table: {table_name}"]
  schema_lines.append("Columns: ")

  for col in columns:
    # col[1]: column name(Second element of the tuple)
    # col[2]: data type(third element)
    schema_lines.append(f"  - {col[1]} ({col[2]})")

  # Add sample values to help AI understand the data
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 5")
  sample_data = cursor.fetchall()
  schema_lines.append("\nSample rows (first 3): ")

  for row in sample_data:
    schema_lines.append(f"  {row}")

  return "\n".join(schema_lines)
  # "\n".join : Joins all lines with a new line
  # This creates a single, multi line string

# Test the schema function
schema = get_schema(conn)
print(schema)

Table: students
Columns: 
  - student_id (INTEGER)
  - name (TEXT)
  - age (INTEGER)
  - gender (TEXT)
  - department (TEXT)
  - semester (INTEGER)
  - math_score (INTEGER)
  - science_score (INTEGER)
  - english_score (INTEGER)
  - programming_score (INTEGER)
  - attendance_percentage (INTEGER)
  - city (TEXT)
  - admission_year (INTEGER)

Sample rows (first 3): 
  (1001, 'Aarav Sharma', 19, 'Male', 'Computer Science', 2, 85, 78, 72, 91, 92, 'Mumbai', 2023)
  (1002, 'Priya Patel', 20, 'Female', 'Computer Science', 2, 76, 82, 88, 79, 87, 'Ahmedabad', 2023)
  (1003, 'Rohit Verma', 19, 'Male', 'Electronics', 2, 65, 74, 61, 55, 78, 'Delhi', 2023)
  (1004, 'Sneha Reddy', 20, 'Female', 'Mechanical', 2, 70, 80, 75, 48, 95, 'Hyderabad', 2023)
  (1005, 'Arjun Nair', 19, 'Male', 'Computer Science', 2, 92, 88, 81, 95, 90, 'Kochi', 2023)


In [ ]:
def generate_sql(user_question, schema_text, client, model):
  """
  Sends the user's quesrions and database schema to Groq model.
  Groq LLM generates a SQL query that answers the user's question.
  PARAMS:
    user_question : Question
    schema_tex: ..
  RETURNS:
    A single string containing the generated SQL query.
  """

  # Define the system prompt = instructions we give
  system_prompt = f"""You are the developer of SQL.
  You are connected to a SQLite database with the following structure:
  {schema_text}

  Rules you must follow:
  1. Generate ONLY a valid SQLite SQL query.
  2. Do not include any explanation or text — only the SQL query.
  3. Do not use markdown code blocks. Return the raw SQL only.
  4. The table name is: students
  5. Only use column names that exist in the schema above.
  6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
  7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
  """

  # call groq api
  response = client.chat.completions.create(
    model=model,
    messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_question}
    ],
    temperature = 0.0
  )

  # Extract the
  sql_query = response.choices[0].message.content.strip()     # strip() -> to remove white spaces

  return sql_query

# This is the system prompt - the set of instructions we give to the AI
# f"""...""": A multi-line f-string (triple quotes allow line breaks)
# {schema_text}: We inject the actual schema text into the prompt
# This is exactly like RAG

In [ ]:
question = "Show me all female students"
print(f"Question: {question}")
print("\nGenerating SQL...")
sql_query = generate_sql(question, schema, client, MODEL)
print(f"SQL Query: {sql_query}")
#

Question: Show me all female students

Generating SQL...
SQL Query: SELECT * FROM students WHERE gender = 'Female'


In [ ]:
question = "Show me number of male and female students"
print(f"Question: {question}")
print("\nGenerating SQL...")
sql_query = generate_sql(question, schema, client, MODEL)
print(f"SQL Query: {sql_query}")
#

Question: Show me number of male and female students

Generating SQL...
SQL Query: SELECT COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count 
FROM students


In [ ]:
def execute_sql(sql_query, conn):
  """
    Cleans the AI generated SQL query and executes it on the SQL database.
    PARAMS:
      sql_query: SQL Query to be cleaned
      conn: Database connection
    RETURNS:
      Returns the dataframe with query results or error message.
  """
  clean_sql = sql_query.strip()
  # strip(): removes white space from both ends
  # clean_sql = re.sub(r'```\s*', '', clean_sql, flags=re.IGNORECASE).strip() -> INSTEAD OF BELOW 2 LINES
  clean_sql = re.sub(r'```\s*', '', clean_sql)
  clean_sql = sql_query.strip()

  try:
    result_df = pd.read_sql_query(clean_sql, conn)
    return result_df, None
  except Exception as e:
    return None, str(e)

print(f"Executing SQL: {sql_query}")
result, error = execute_sql(sql_query, conn)
if error:
  print(f"Error executing SQL: {error}")
else:
  print(f"\nQuery returned {len(result)} rows")

Executing SQL: SELECT COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count 
FROM students

Query returned 1 rows


In [ ]:
def text_to_sql_agent(user_question, conn, client, model, verbose=True):
  """
  The main AI agent function.
  """
  print(f"USER QUESTION: {user_question}")
  print('='*40)

# STEP 1: GET THE DATABASE SCHEMA
  if verbose:
    print("\nSTEP 1: Reading database schema")

  schema_text=get_schema(conn)
  # this gives the AI map of our Database
  if verbose:
    print("Schema loaded  successfully")

  # 2. Generate SQL
  if verbose:
    print("\nSTEP 2: Generating SQL query with Groq LLM")

  generated_sql = generate_sql(user_question, schema_text, client, model)

  # 3. Execute SQL on database
  if verbose:
    print("\n STEP 3: Executing SQL in the database")

  result_df, error = execute_sql(generated_sql, conn)

  # 4. Display the result
  if verbose:
    print(f"\nSTEP 4: Query returned {len(result_df)} row(s)")
    print("\nRESULTS:")
    print('-'*20)
    print(result_df.to_string(index=False))

  print("="*30)
  return result_df, generated_sql

# test our commplete agent:
result, sql_used = text_to_sql_agent("Show top 5 students in programming", conn, client, MODEL)

USER QUESTION: Show top 5 students in programming

STEP 1: Reading database schema
Schema loaded  successfully

STEP 2: Generating SQL query with Groq LLM

 STEP 3: Executing SQL in the database

STEP 4: Query returned 5 row(s)

RESULTS:
--------------------
          name
    Ananya Das
   Tanvi Mehta
    Arjun Nair
Akanksha Yadav
   Divya Singh


In [ ]:
### function to generate NLP

def generate_nlp(user_question, query_result_df, client, model, verbose=False):
  """ generate an text response that answers user's question in NLP
  user_question:typed by the user
  query_result_df: the pandas DataFrame containing the query results
  """
  #define system prompt
  system_prompt=f"""You are an expert in natural language processing. Your task is to generate a concise, human-readable answer to the user's question based on the provided query results.



Rules you must follow:
1. Start the response with a rephrased version of the user's question in the first person narration, past tense, and passive voice.
2. Do not mention anything about rephrasing or narration style in your response.
3. Do not include any explanations or conversational filler.
4. Return only the answer in complete sentences. For lists of items (like names), present each item on a new line.
5. Do not return a DataFrame or any code block.
6. Only refer to column names and values present in the 'Query Result Data' provided above.
7. Do not include any markdown code blocks.
"""

  ## call the groq API
  response = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_question}
      ],
      temperature = 0.3
  )

  nlp_response = response.choices[0].message.content.strip()
  if verbose:
    print(f'\n[STEP 5] NLP ANSWER')
    print("\nRESPONSE:")
    print("-"*60)
    print(nlp_response)
  return nlp_response